# Train baseline MobileNetV2 - phan loai tuoi/hong nong san

Mo notebook nay bang Google Colab (chinh la cach dung nhat neu ban dang doc file .ipynb tren GitHub: bam nut 'Open in Colab' hoac vao colab.research.google.com > File > Upload notebook).

Chay TUAN TU tung o tu tren xuong (Shift+Enter). Khong can cai gi tren may ca nhan.

**Truoc tien: bat GPU** - menu Runtime > Change runtime type > Hardware accelerator > GPU > Save. Roi chay o duoi day de kiem tra.

In [ ]:
import tensorflow as tf

gpus = tf.config.list_physical_devices('GPU')
print('GPU:', gpus if gpus else 'KHONG co GPU - vao Runtime > Change runtime type > chon GPU roi chay lai o tren')

## Buoc 1: Tai dataset tu Kaggle

Dataset dung: **Fruits fresh and rotten for classification** (mien phi, da gan nhan san: tao/chuoi/cam x tuoi/hong).

Can file `kaggle.json` de xac thuc voi Kaggle (chi lam 1 lan, dung cho moi lan chay notebook sau nay):
1. Vao https://www.kaggle.com/settings
2. Muc **API** > bam **Create New Token** > file `kaggle.json` se tu tai ve may ban
3. Chay o duoi day, bam **Choose Files** va chon dung file `kaggle.json` vua tai

In [ ]:
from google.colab import files

print('Chon file kaggle.json vua tai tu kaggle.com/settings (muc API > Create New Token)')
uploaded = files.upload()

!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json
!pip -q install kaggle
!kaggle datasets download -d sriramr/fruits-fresh-and-rotten-for-classification -p /content
!unzip -q -o /content/fruits-fresh-and-rotten-for-classification.zip -d /content/data
print('Da tai va giai nen xong.')

## Buoc 2: Xem cay thu muc thuc te

Moi ban Kaggle co the nen file hoi khac nhau ve ten thu muc goc, nen chay o duoi day de xem CHINH XAC duong dan tren may ban, roi dung ket qua nay de dien lai `TRAIN_DIR` / `TEST_DIR` o Buoc 3.

In [ ]:
import pathlib

data_root = pathlib.Path('/content/data')
for p in sorted(data_root.rglob('*')):
    if p.is_dir():
        depth = len(p.relative_to(data_root).parts)
        if depth <= 2:
            n_images = len(list(p.glob('*.*')))
            label = '  ' * depth + p.name
            if n_images:
                label += f'  ({n_images} anh)'
            print(label)

## Buoc 3: Nap du lieu

**Sua 2 duong dan `TRAIN_DIR` va `TEST_DIR` o duoi day cho khop voi cay thu muc in ra o Buoc 2** (thuong se co dang `/content/data/<mot-thu-muc>/train` va `.../test`, ben trong moi thu muc do lai co cac thu muc con la ten lop nhu `freshapples`, `rottenbanana`...).

`image_dataset_from_directory` se tu dong xem moi thu muc con la 1 lop, va gan nhan theo THU TU BANG CHU CAI cua ten thu muc - do la ly do can in `class_names` ra o cuoi o nay va dung dung thu tu do khi cap nhat file `classifier.py` sau nay.

In [ ]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 32

# SUA 2 dong duoi day cho khop voi ket qua in ra o Buoc 2
TRAIN_DIR = '/content/data/dataset/train'
TEST_DIR = '/content/data/dataset/test'

train_ds = tf.keras.utils.image_dataset_from_directory(
    TRAIN_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical')
test_ds = tf.keras.utils.image_dataset_from_directory(
    TEST_DIR, image_size=IMG_SIZE, batch_size=BATCH_SIZE, label_mode='categorical', shuffle=False)

CLASS_NAMES = train_ds.class_names
print('Danh sach lop (GHI LAI dung THU TU nay, se can dan vao ai/app/classifier.py o Buoc cuoi):')
print(CLASS_NAMES)

## Buoc 4: Tien xu ly anh

MobileNetV2 doi hoi anh duoc dua ve khoang gia tri [-1, 1] theo mot cong thuc rieng (khac voi chia don gian cho 255). Buoc nay PHAI khop voi cach `ai/app/classifier.py` xu ly anh khi phuc vu du doan that - va file do da duoc viet san de dung dung ham nay (`tf.keras.applications.mobilenet_v2.preprocess_input`), nen o day chi can dung dung ham nay la se dong bo.

In [ ]:
preprocess_input = tf.keras.applications.mobilenet_v2.preprocess_input

def prep(x, y):
    return preprocess_input(x), y

train_ds = train_ds.map(prep).prefetch(tf.data.AUTOTUNE)
test_ds = test_ds.map(prep).prefetch(tf.data.AUTOTUNE)

## Buoc 5: Dung model (Transfer Learning)

- `MobileNetV2(weights='imagenet')`: phan 'da biet nhin' co san, dong bang (`trainable = False`) - khong day lai phan nay.
- `RandomFlip` / `RandomRotation`: xoay/lat anh ngau nhien luc train de model khong 'hoc vet' goc chup, giup do chinh xac thuc te tot hon.
- `GlobalAveragePooling2D + Dense(softmax)`: phan MOI duoc them vao va se duoc day de phan biet cac lop tuoi/hong.

In [ ]:
NUM_CLASSES = len(CLASS_NAMES)

base_model = tf.keras.applications.MobileNetV2(
    input_shape=IMG_SIZE + (3,), include_top=False, weights='imagenet')
base_model.trainable = False

inputs = tf.keras.Input(shape=IMG_SIZE + (3,))
x = tf.keras.layers.RandomFlip('horizontal')(inputs)
x = tf.keras.layers.RandomRotation(0.1)(x)
x = base_model(x, training=False)
x = tf.keras.layers.GlobalAveragePooling2D()(x)
x = tf.keras.layers.Dropout(0.2)(x)
outputs = tf.keras.layers.Dense(NUM_CLASSES, activation='softmax')(x)
model = tf.keras.Model(inputs, outputs)

model.compile(optimizer=tf.keras.optimizers.Adam(1e-3),
              loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

## Buoc 6: Huan luyen (giai doan 1)

Chi day phan moi them vao, base model van dong bang. Voi Kaggle GPU/Colab GPU mien phi, 10 epoch tren dataset nay thuong mat khoang 10-20 phut.

In [ ]:
history = model.fit(train_ds, validation_data=test_ds, epochs=10)

## Buoc 7 (tuy chon, nen lam neu con thoi gian): Fine-tune

Mo khoa vai lop CUOI CUNG cua MobileNetV2 de tinh chinh sau hon, dung learning rate rat nho (1e-5) de khong 'pha' kien thuc no da hoc san tu ImageNet. Buoc nay giup tang do chinh xac them vai %, khong bat buoc cho baseline tuan 3.

In [ ]:
base_model.trainable = True
for layer in base_model.layers[:-20]:
    layer.trainable = False

model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
              loss='categorical_crossentropy', metrics=['accuracy'])
history_finetune = model.fit(train_ds, validation_data=test_ds, epochs=5)

## Buoc 8: Danh gia

Do chinh xac tren tap test (anh model CHUA tung thay luc train) - day la con so nen dua vao bao cao do an, khong phai do chinh xac tren tap train (thuong cao hon that te).

In [ ]:
loss, acc = model.evaluate(test_ds)
print(f'Do chinh xac tren tap test: {acc * 100:.2f}%')

## Buoc 9: Luu va tai model ve may

In [ ]:
model.save('produce_classifier.keras')

from google.colab import files
files.download('produce_classifier.keras')

print('Sau khi tai xong: copy file nay vao ai/models/produce_classifier.keras trong repo (thu muc da gitignore, chia se qua Drive/USB cho nhom, khong push len Git vi file nang).')
print()
print('Danh sach LABELS can dan de vao ai/app/classifier.py (dung DUNG THU TU):')
print(CLASS_NAMES)

## Checklist sau khi xong

1. Copy `produce_classifier.keras` vao `ai/models/produce_classifier.keras`
2. Mo `ai/app/classifier.py`, thay bien `LABELS` bang danh sach in ra o Buoc 9 (dung dung thu tu)
3. Tren may that (khong phai Colab): `cd ai && pip install -r requirements-model.txt`
4. Chay `uvicorn app.main:app --reload --port 8000`, mo `http://localhost:8000/health` phai thay `model_loaded: true`
5. Thu goi `POST /classify` voi 1 anh that de kiem tra ket qua co hop ly khong (dung Swagger cua backend hoac Postman goi thang qua AI service o buoc test nay)